# Play Pacman With Coins

Choose an environment with the buttons, preview an `rgb_array` frame inline, then run the last cell to play with the arrow keys in a pygame window.

In [ ]:
import ipywidgets as widgets
from IPython.display import display
from PIL import Image

VALID_ENV_NAMES = ("mini_pacman_with_coins", "pacman_with_coins")

def make_env(env_name, **kwargs):
    if env_name == "mini_pacman_with_coins":
        from masa.envs.discrete.mini_pacman_with_coins import MiniPacmanWithCoins

        return MiniPacmanWithCoins(**kwargs)
    if env_name == "pacman_with_coins":
        from masa.envs.discrete.pacman_with_coins import PacmanWithCoins

        return PacmanWithCoins(**kwargs)
    raise ValueError(f"env_name must be one of {VALID_ENV_NAMES!r}")

ENV_SELECTOR = widgets.ToggleButtons(
    options=VALID_ENV_NAMES,
    value="mini_pacman_with_coins",
    description="Env",
)
display(ENV_SELECTOR)

ENV_NAME = ENV_SELECTOR.value
PACMAN_HAT = "crown"
GHOST_COLORS = ((71, 202, 255),)
SEED = 0


In [ ]:
ENV_NAME = ENV_SELECTOR.value
preview_env = make_env(
    ENV_NAME,
    render_mode="rgb_array",
    render_window_size=512,
    pacman_hat=PACMAN_HAT,
    ghost_colors=GHOST_COLORS,
)
preview_env.reset(seed=SEED)
frame = preview_env.render()
display(Image.fromarray(frame))
preview_env.close()


In [ ]:
def play(env_name=None, seed=SEED):
    import pygame

    if env_name is None:
        env_name = ENV_SELECTOR.value
    action_keys = {
        pygame.K_LEFT: 0,
        pygame.K_RIGHT: 1,
        pygame.K_DOWN: 2,
        pygame.K_UP: 3,
        pygame.K_SPACE: 4,
    }
    env = make_env(
        env_name,
        render_mode="human",
        render_window_size=512,
        pacman_hat=PACMAN_HAT,
        ghost_colors=GHOST_COLORS,
    )
    env.reset(seed=seed)
    running = True
    while running and not env.human_window_closed:
        chosen_action = None
        for event in pygame.event.get():
            if not env.handle_pygame_event(event):
                running = False
                break
            if event.type == pygame.KEYDOWN:
                if event.key == pygame.K_ESCAPE:
                    running = False
                    break
                chosen_action = action_keys.get(event.key, chosen_action)
        if chosen_action is None:
            env.render()
            continue
        _, reward, terminated, truncated, _ = env.step(chosen_action)
        if reward:
            print(f"reward={reward}")
        if terminated or truncated:
            env.reset(seed=seed)
    env.close()

play()
